# 01 — Data Collection

This notebook documents and executes the data collection process for the Medicaid expansion causal inference analysis.

**Data sources:**
1. Medicaid Expansion Status (KFF) — manually compiled
2. State-level controls (ACS via Census API)
3. Mortality data (CDC WONDER) — manual query instructions
4. Diabetes outcomes (CDC Diabetes Atlas)
5. Maternal/infant health (CDC WONDER Natality)
6. Health access measures (BRFSS)

**Output:** Raw CSV files saved to `data/raw/`

In [1]:
import pandas as pd
import numpy as np
import os
import requests
from io import StringIO

# Create directories if they don't exist
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print("Directories ready.")

Directories ready.


---
## 1. Medicaid Expansion Status (Treatment Variable)

Source: [Kaiser Family Foundation](https://www.kff.org/medicaid/issue-brief/status-of-state-medicaid-expansion-decisions-interactive-map/)

We manually compile the expansion status and dates. This is the most reliable approach since KFF doesn't provide a clean API.

In [2]:
# Medicaid expansion data — compiled from KFF (verify dates against latest KFF data)
# Format: state, FIPS code, expansion year (0 = never expanded), effective date

expansion_data = {
    'state': [
        'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
        'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida',
        'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
        'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine',
        'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
        'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
        'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota',
        'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island',
        'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah',
        'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin',
        'Wyoming'
    ],
    'state_fips': [
        '01', '02', '04', '05', '06',
        '08', '09', '10', '11', '12',
        '13', '15', '16', '17', '18',
        '19', '20', '21', '22', '23',
        '24', '25', '26', '27', '28',
        '29', '30', '31', '32', '33',
        '34', '35', '36', '37', '38',
        '39', '40', '41', '42', '44',
        '45', '46', '47', '48', '49',
        '50', '51', '53', '54', '55',
        '56'
    ],
    # Expansion year: 0 = never expanded as of 2024
    # IMPORTANT: Verify these against latest KFF data before running analysis
    'expansion_year': [
        0, 2015, 2014, 2014, 2014,        # AL, AK, AZ, AR, CA
        2014, 2014, 2014, 2014, 0,         # CO, CT, DE, DC, FL
        0, 2014, 2020, 2014, 2015,         # GA, HI, ID, IL, IN
        2014, 0, 2014, 2016, 2019,         # IA, KS, KY, LA, ME
        2014, 2014, 2014, 2014, 0,         # MD, MA, MI, MN, MS
        2021, 2016, 2020, 2014, 2014,      # MO, MT, NE, NV, NH
        2014, 2014, 2014, 2023, 2014,      # NJ, NM, NY, NC, ND
        2014, 2021, 2014, 2015, 2014,      # OH, OK, OR, PA, RI
        0, 2023, 0, 0, 2020,               # SC, SD, TN, TX, UT
        2014, 2019, 2014, 2014, 0,         # VT, VA, WA, WV, WI
        0                                   # WY
    ],
    'expansion_date': [
        None, '2015-09-01', '2014-01-01', '2014-01-01', '2014-01-01',
        '2014-01-01', '2014-01-01', '2014-01-01', '2014-01-01', None,
        None, '2014-01-01', '2020-01-01', '2014-01-01', '2015-02-01',
        '2014-01-01', None, '2014-01-01', '2016-07-01', '2019-01-10',
        '2014-01-01', '2014-01-01', '2014-04-01', '2014-01-01', None,
        '2021-07-01', '2016-01-01', '2020-10-01', '2014-01-01', '2014-08-15',
        '2014-01-01', '2014-01-01', '2014-01-01', '2023-12-01', '2014-01-01',
        '2014-01-01', '2021-07-01', '2014-01-01', '2015-01-01', '2014-01-01',
        None, '2023-07-01', None, None, '2020-01-01',
        '2014-01-01', '2019-01-01', '2014-01-01', '2014-01-01', None,
        None
    ]
}

df_expansion = pd.DataFrame(expansion_data)

# Create derived variables
df_expansion['ever_expanded'] = (df_expansion['expansion_year'] > 0).astype(int)
df_expansion['cohort'] = df_expansion['expansion_year'].apply(
    lambda x: 'Never' if x == 0 else ('Early (2014)' if x == 2014 else f'Late ({x})')
)

print(f"Total states + DC: {len(df_expansion)}")
print(f"\nExpansion status:")
print(df_expansion['ever_expanded'].value_counts().rename({1: 'Expanded', 0: 'Not expanded'}))
print(f"\nBy cohort:")
print(df_expansion['cohort'].value_counts().sort_index())

df_expansion.to_csv('../data/raw/kff_expansion_status.csv', index=False)
print("\nSaved to data/raw/kff_expansion_status.csv")

Total states + DC: 51

Expansion status:
ever_expanded
Expanded        41
Not expanded    10
Name: count, dtype: int64

By cohort:
cohort
Early (2014)    27
Late (2015)      3
Late (2016)      2
Late (2019)      2
Late (2020)      3
Late (2021)      2
Late (2023)      2
Never           10
Name: count, dtype: int64

Saved to data/raw/kff_expansion_status.csv


In [3]:
# Quick look at the expansion data
df_expansion[['state', 'expansion_year', 'cohort']].sort_values('expansion_year')

,state,expansion_year,cohort
0,Alabama,0,Never
9,Florida,0,Never
10,Georgia,0,Never
16,Kansas,0,Never
24,Mississippi,0,Never
40,South Carolina,0,Never
43,Texas,0,Never
42,Tennessee,0,Never
49,Wisconsin,0,Never
50,Wyoming,0,Never


---
## 2. State-Level Controls (American Community Survey)

We pull state-level demographic and economic controls from the Census Bureau API.

**You need a free Census API key:** https://api.census.gov/data/key_signup.html

Once you get it, paste it below.

In [8]:
# ============================================
# PASTE YOUR CENSUS API KEY HERE
# Get one free at: https://api.census.gov/data/key_signup.html
# ============================================
CENSUS_API_KEY = 'e2bdcde4803318f60bde3f92c99ff12895f556be'

In [9]:
def fetch_acs_data(year, api_key):
    """
    Fetch state-level ACS 1-year estimates for a given year.
    
    Variables pulled:
    - B19013_001E: Median household income
    - B17001_001E: Total population (poverty universe)
    - B17001_002E: Population below poverty level
    - B27001_001E: Total population (insurance universe)
    - B01003_001E: Total population
    - B03002_003E: White non-Hispanic
    - B03002_004E: Black non-Hispanic  
    - B03002_012E: Hispanic/Latino
    """
    
    base_url = f'https://api.census.gov/data/{year}/acs/acs1'
    
    variables = [
        'NAME',
        'B19013_001E',   # Median household income
        'B17001_001E',   # Poverty universe total
        'B17001_002E',   # Below poverty
        'B01003_001E',   # Total population
        'B03002_003E',   # White non-Hispanic
        'B03002_004E',   # Black non-Hispanic
        'B03002_012E',   # Hispanic/Latino
    ]
    
    params = {
        'get': ','.join(variables),
        'for': 'state:*',
        'key': api_key
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        df = pd.DataFrame(data[1:], columns=data[0])
        df['year'] = year
        
        # Rename columns
        df = df.rename(columns={
            'NAME': 'state',
            'B19013_001E': 'median_household_income',
            'B17001_001E': 'poverty_universe',
            'B17001_002E': 'below_poverty',
            'B01003_001E': 'total_population',
            'B03002_003E': 'white_non_hispanic',
            'B03002_004E': 'black_non_hispanic',
            'B03002_012E': 'hispanic_latino',
            'state': 'state_fips'
        })
        
        # Convert to numeric
        numeric_cols = ['median_household_income', 'poverty_universe', 'below_poverty',
                        'total_population', 'white_non_hispanic', 'black_non_hispanic',
                        'hispanic_latino']
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Calculate rates
        df['poverty_rate'] = (df['below_poverty'] / df['poverty_universe'] * 100).round(2)
        df['pct_white'] = (df['white_non_hispanic'] / df['total_population'] * 100).round(2)
        df['pct_black'] = (df['black_non_hispanic'] / df['total_population'] * 100).round(2)
        df['pct_hispanic'] = (df['hispanic_latino'] / df['total_population'] * 100).round(2)
        
        print(f"  {year}: {len(df)} states fetched")
        return df
        
    except Exception as e:
        print(f"  {year}: ERROR — {e}")
        return None

In [10]:
# Fetch ACS data for all years (2010-2022)
# Note: ACS 1-year was not released in 2020 due to COVID data collection issues

years = list(range(2010, 2023))  # 2010 through 2022
print("Fetching ACS data...")

acs_frames = []
for year in years:
    df = fetch_acs_data(year, CENSUS_API_KEY)
    if df is not None:
        acs_frames.append(df)

if acs_frames:
    df_acs = pd.concat(acs_frames, ignore_index=True)
    
    # Keep clean columns
    keep_cols = ['state', 'state_fips', 'year', 'total_population', 
                 'median_household_income', 'poverty_rate',
                 'pct_white', 'pct_black', 'pct_hispanic']
    df_acs = df_acs[keep_cols]
    
    df_acs.to_csv('../data/raw/acs_state_controls.csv', index=False)
    print(f"\nSaved {len(df_acs)} state-year observations to data/raw/acs_state_controls.csv")
    print(f"Years covered: {df_acs['year'].min()} - {df_acs['year'].max()}")
    print(f"States per year: ~{df_acs.groupby('year').size().median():.0f}")
else:
    print("\nNo data fetched. Check your API key.")

Fetching ACS data...
  2010: ERROR — Expecting value: line 2 column 1 (char 1)
  2011: ERROR — Expecting value: line 2 column 1 (char 1)
  2012: ERROR — Expecting value: line 2 column 1 (char 1)
  2013: ERROR — Expecting value: line 2 column 1 (char 1)
  2014: ERROR — Expecting value: line 2 column 1 (char 1)
  2015: ERROR — Expecting value: line 2 column 1 (char 1)
  2016: ERROR — Expecting value: line 2 column 1 (char 1)
  2017: ERROR — Expecting value: line 2 column 1 (char 1)
  2018: ERROR — Expecting value: line 2 column 1 (char 1)
  2019: ERROR — Expecting value: line 2 column 1 (char 1)
  2020: ERROR — Expecting value: line 2 column 1 (char 1)
  2021: ERROR — Expecting value: line 2 column 1 (char 1)
  2022: ERROR — Expecting value: line 2 column 1 (char 1)

No data fetched. Check your API key.


In [11]:
# Quick validation
if 'df_acs' in dir():
    print("Sample data (2014):")
    display(df_acs[df_acs['year'] == 2014].head(10))
    
    print("\nMissing values by column:")
    print(df_acs.isnull().sum())

---
## 3. CDC WONDER — Mortality Data (Manual Download)

CDC WONDER requires interactive queries through their web interface. Below are step-by-step instructions for each query.

### 3a. All-Cause Mortality

1. Go to: https://wonder.cdc.gov/mcd.html
2. Click "I Agree" on the data use agreement
3. Set the following parameters:
   - **Group Results By:** State, Year
   - **Show Totals:** checked
   - **Show Zero Values:** checked  
   - **Year/Month:** Select years 2010-2022
   - **All other filters:** leave as default (all causes, all ages, etc.)
4. Click "Send" → "Export Results" → Save as `cdc_wonder_allcause_mortality.txt`
5. Move the file to `data/raw/`

### 3b. Diabetes-Related Mortality

Same as above, but add:
- **ICD-10 Codes:** E10-E14 (Diabetes mellitus)
- Under "Select cause of death": search for "Diabetes" and select all E10-E14 codes
- Save as `cdc_wonder_diabetes_mortality.txt`

### 3c. Maternal Mortality

Same setup, but:
- **ICD-10 Codes:** O00-O99 (Pregnancy, childbirth, puerperium) + A34 (Obstetrical tetanus)
- **Sex:** Female
- **Age Groups:** 15-44 (or leave all)
- Save as `cdc_wonder_maternal_mortality.txt`

In [12]:
def parse_cdc_wonder_txt(filepath):
    """
    Parse CDC WONDER tab-delimited export files.
    CDC WONDER exports include metadata rows at the bottom that need to be removed.
    """
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        print("Please download from CDC WONDER following the instructions above.")
        return None
    
    # Read the file, stopping at the notes section
    rows = []
    with open(filepath, 'r') as f:
        header = f.readline().strip().split('\t')
        for line in f:
            if line.startswith('---') or line.startswith('"---'):
                break  # Stop at the notes/metadata section
            if line.strip():
                rows.append(line.strip().split('\t'))
    
    df = pd.DataFrame(rows, columns=header)
    
    # Clean up: remove quotes, convert numeric columns
    for col in df.columns:
        df[col] = df[col].str.strip('"')
    
    # Standard CDC WONDER columns
    if 'Deaths' in df.columns:
        df['Deaths'] = pd.to_numeric(df['Deaths'].replace('Suppressed', np.nan), errors='coerce')
    if 'Population' in df.columns:
        df['Population'] = pd.to_numeric(df['Population'], errors='coerce')
    if 'Crude Rate' in df.columns:
        df['Crude Rate'] = pd.to_numeric(df['Crude Rate'].replace(['Suppressed', 'Unreliable'], np.nan), errors='coerce')
    if 'Age Adjusted Rate' in df.columns:
        df['Age Adjusted Rate'] = pd.to_numeric(
            df['Age Adjusted Rate'].replace(['Suppressed', 'Unreliable', 'Not Applicable'], np.nan), 
            errors='coerce'
        )
    
    print(f"Parsed {filepath}: {len(df)} rows, {df.columns.tolist()}")
    return df

In [13]:
# Parse mortality files (run after downloading from CDC WONDER)

mortality_files = {
    'allcause': '../data/raw/cdc_wonder_allcause_mortality.txt',
    'diabetes': '../data/raw/cdc_wonder_diabetes_mortality.txt',
    'maternal': '../data/raw/cdc_wonder_maternal_mortality.txt'
}

for name, filepath in mortality_files.items():
    df = parse_cdc_wonder_txt(filepath)
    if df is not None:
        print(f"\n{name} mortality — first 5 rows:")
        display(df.head())

File not found: ../data/raw/cdc_wonder_allcause_mortality.txt
Please download from CDC WONDER following the instructions above.
File not found: ../data/raw/cdc_wonder_diabetes_mortality.txt
Please download from CDC WONDER following the instructions above.
File not found: ../data/raw/cdc_wonder_maternal_mortality.txt
Please download from CDC WONDER following the instructions above.


---
## 4. CDC Diabetes Surveillance Data

### Manual Download Instructions

1. Go to: https://gis.cdc.gov/grasp/diabetes/diabetesatlas-surveillance.html
2. Select **"Prevalence"** tab → download state-level data for all years
3. Repeat for **"Incidence"** and **"Mortality"** tabs
4. Save files to `data/raw/` as:
   - `cdc_diabetes_prevalence.csv`
   - `cdc_diabetes_incidence.csv`
   - `cdc_diabetes_mortality.csv`

Alternatively, check if bulk download is available from: https://gis.cdc.gov/grasp/diabetes/diabetesatlas.html

In [ ]:
# Parse diabetes data (run after downloading)
diabetes_files = [
    '../data/raw/cdc_diabetes_prevalence.csv',
    '../data/raw/cdc_diabetes_incidence.csv',
    '../data/raw/cdc_diabetes_mortality.csv'
]

for filepath in diabetes_files:
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"\n{filepath}:")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"Not yet downloaded: {filepath}")

---
## 5. CDC WONDER — Natality (Maternal & Infant Health)

### Manual Download Instructions

1. Go to: https://wonder.cdc.gov/natality.html
2. Accept data use agreement
3. Set parameters:
   - **Group Results By:** State, Year
   - **Years:** 2010-2022
   - **Measures to include:** Check all available (births, birth weight, prenatal care, etc.)
4. Click "Send" → "Export Results"
5. Save as `cdc_wonder_natality.txt` in `data/raw/`

In [ ]:
# Parse natality data (run after downloading)
natality_path = '../data/raw/cdc_wonder_natality.txt'

df_natality = parse_cdc_wonder_txt(natality_path)
if df_natality is not None:
    display(df_natality.head())

---
## 6. BRFSS Health Access Data

### Option A: Pre-computed state-level prevalence (Recommended)

1. Go to: https://www.cdc.gov/brfss/brfssprevalence/index.html
2. Select indicator: "Could Not See Doctor Due to Cost" (or others)
3. Select all states, years 2010-2022
4. Export and save as `brfss_health_access.csv`

### Option B: CDC API (SODAPI)

Some BRFSS data is available through CDC's SODAPI. Below we try to pull it programmatically.

In [ ]:
# Attempt to pull BRFSS data via CDC SODAPI
# This endpoint may change — verify at https://data.cdc.gov

brfss_url = 'https://data.cdc.gov/resource/dttw-5yxu.json'

try:
    params = {
        '$limit': 50000,
        '$where': "year >= '2010'",
    }
    response = requests.get(brfss_url, params=params)
    
    if response.status_code == 200:
        df_brfss = pd.DataFrame(response.json())
        print(f"BRFSS data: {df_brfss.shape}")
        print(f"Columns: {df_brfss.columns.tolist()[:10]}...")
        df_brfss.to_csv('../data/raw/brfss_health_access.csv', index=False)
        print("Saved to data/raw/brfss_health_access.csv")
    else:
        print(f"API returned status {response.status_code}")
        print("Please download manually from CDC BRFSS Prevalence tool.")
        
except Exception as e:
    print(f"Could not fetch BRFSS data: {e}")
    print("Please download manually from CDC BRFSS Prevalence tool.")

---
## Data Collection Summary

| Dataset | Method | Status |
|---------|--------|--------|
| Medicaid Expansion Status | Manual compilation | ✅ Done |
| ACS State Controls | Census API | ✅ / ❌ (check above) |
| All-Cause Mortality | CDC WONDER manual | ⬜ Download needed |
| Diabetes Mortality | CDC WONDER manual | ⬜ Download needed |
| Maternal Mortality | CDC WONDER manual | ⬜ Download needed |
| Diabetes Surveillance | CDC Atlas manual | ⬜ Download needed |
| Natality Data | CDC WONDER manual | ⬜ Download needed |
| BRFSS Health Access | CDC API / manual | ✅ / ❌ (check above) |

### Next Step
Once all raw data is downloaded, proceed to **`02_data_cleaning.ipynb`** to merge everything into the analysis panel.

In [ ]:
# Final check: what files do we have in data/raw?
raw_files = os.listdir('../data/raw')
print("Files in data/raw/:")
for f in sorted(raw_files):
    if not f.startswith('.'):
        size = os.path.getsize(f'../data/raw/{f}') / 1024
        print(f"  {f} ({size:.1f} KB)")